# CoSE 220 · AI & Innovation — the day notebook

One notebook for the whole day. **Part N goes with deck D01.0N**:

| Part | Deck | Time | What you do |
| --- | --- | --- | --- |
| 1 · Get Ready | D01.01 | 9:30 | check your laptop works |
| 2 · Train, Then Overfit | D01.02 | 10:00 | fit a line, then a network that sees — and overfits |
| 3 · How an LLM Reads | D01.03 | 11:15 | tokens, next words, meaning as numbers |
| 4 · Build a RAG | D01.04 | 1:00 | an AI that answers from a public document |
| 5 · Diagnose and Retest | D01.05 | 2:15 | three questions, one change, your demo card |

**How to use it:** open the table of contents (☰ on the left), run one part at a time, top to
bottom, with each cell's ▶ button. **If Colab disconnects**, re-run the ⚙️ SETUP cell of your part.

| Marker | Means |
| --- | --- |
| ▶ DEMO | just run it |
| 🔍 READ | **predict first**, then run |
| ✍️ YOU TRY | change something — the boxes and sliders on the right |
| ✅ CHECKPOINT | tells you whether you got it |

Stuck? Select a cell and ask the AI assistant to explain it.

# Part 1 · Get Ready
**Deck D01.01** · 9:30

1. **File → Save a copy in Drive.** Use *your copy* all day — it keeps your changes.
2. Make sure you are **signed in to Google** (top right).
3. Run the cell below with its ▶ button. It takes about a minute and ends with **✅ READY FOR CLASS**.

**Something red?** A work laptop that blocks Google services is the usual cause — share your
partner's laptop for today.

In [ ]:
# ▶ DEMO 1.1.1 — check your laptop (about a minute)
import importlib.util, subprocess, sys
for package, module in (("sentence-transformers", "sentence_transformers"), ("pypdf", "pypdf")):
    if importlib.util.find_spec(module) is None:
        print(f"installing {package} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import numpy, matplotlib, sklearn, ipywidgets, pypdf
from sentence_transformers import SentenceTransformer

_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
_a, _b, _c = _model.encode(["cut carbon emissions", "reduce CO2 output", "quarterly earnings call"],
                           normalize_embeddings=True)
if float(_a @ _b) > float(_a @ _c):
    print("✅ READY FOR CLASS")
else:
    print("❌ Something is off — show this to your instructor.")

# Part 2 · Train, Then Overfit
**Deck D01.02** · 10:00

Twelve days of ice-cream sales: you fit a line, then the computer does. Then a small network that
learns to see — and what happens when it has too little to learn from.

In [ ]:
# @title ⚙️ PART 2 SETUP — run this first, and again if Colab reconnects { display-mode: "form" }
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from ipywidgets import interact, FloatSlider, fixed

temp_f = np.array([58, 62, 64, 67, 70, 72, 75, 77, 80, 84, 88, 93])   # afternoon high, °F
cones  = np.array([95, 160, 150, 210, 230, 290, 310, 355, 340, 430, 470, 540])   # cones sold that day


def total_squares(m, b):
    """The score: square every gap, add them up."""
    gaps = cones - (m * temp_f + b)
    return float((gaps ** 2).sum())


def draw_line(m, b, title=""):
    """The data, a line, and every gap drawn as a gold square."""
    fig, ax = plt.subplots(figsize=(8, 5))
    xs = np.array([50, 100])
    ax.plot(xs, m * xs + b, color="crimson", lw=3)
    for x, y in zip(temp_f, cones):
        guess = m * x + b
        gap = abs(y - guess)
        ax.add_patch(Rectangle((x, min(y, guess)), gap * (50 / 650) * (5 / 8), gap,
                               color="gold", alpha=0.45, lw=0))
        ax.plot([x, x], [y, guess], color="black", lw=1)
    ax.scatter(temp_f, cones, s=70, zorder=3)
    ax.set_xlim(50, 100); ax.set_ylim(0, 650)
    ax.set_xlabel("Afternoon high (°F)"); ax.set_ylabel("Cones sold")
    ax.set_title(f"{title}   total of the squares = {total_squares(m, b):,.0f}")
    plt.show()


def train(lr, steps=100):
    """Gradient descent from a flat line at zero. Returns m, b and the total after every step.

    Behind the scenes the temperatures are rescaled so that ONE learning rate suits both
    numbers; m and b are converted back to cones-per-degree before they are returned.
    """
    mx, sx = temp_f.mean(), temp_f.std()
    xs = (temp_f - mx) / sx
    w = c = 0.0
    history = []
    for _ in range(steps):
        err = (w * xs + c) - cones
        history.append(float((err ** 2).sum()))
        if history[-1] > 1e12:            # it blew up, stop before the numbers overflow
            break
        w -= lr * 2 * np.mean(err * xs)   # step downhill for the slope...
        c -= lr * 2 * np.mean(err)        # ...and for the intercept
    history.append(float((((w * xs + c) - cones) ** 2).sum()))
    return w / sx, c - w * mx / sx, history


def show_training(lr, moments):
    """The same training run, frozen at a few moments, plus the loss curve."""
    fig, axes = plt.subplots(1, len(moments) + 1, figsize=(4 * (len(moments) + 1), 3.6))
    for ax, k in zip(axes, moments):
        m, b, h = train(lr, steps=k)
        ax.scatter(temp_f, cones, s=25)
        ax.plot([50, 100], [m * 50 + b, m * 100 + b], color="crimson", lw=2.5)
        ax.set_ylim(-50, 650); ax.set_title(f"after {k} steps\ntotal {h[-1]:,.0f}", fontsize=10)
    _, _, h = train(lr, steps=max(moments))
    axes[-1].semilogy(h, lw=2.5); axes[-1].set_title("total of the squares\n(log scale)", fontsize=10)
    axes[-1].set_xlabel("step")
    plt.tight_layout(); plt.show()


def show_prediction(temperature, m, b):
    fig, ax = plt.subplots(figsize=(9, 5))
    xs = np.array([0, 120])
    ax.axvspan(temp_f.min(), temp_f.max(), color="green", alpha=0.08, label="where we have data")
    ax.plot(xs, m * xs + b, color="crimson", lw=2.5)
    ax.axhline(0, color="black", lw=0.8)
    ax.scatter(temp_f, cones, s=40, zorder=3)
    ax.scatter([temperature], [m * temperature + b], s=200, marker="*", color="black", zorder=4,
               label=f"prediction at {temperature}°F")
    ax.set_xlabel("Afternoon high (°F)"); ax.set_ylabel("Cones sold"); ax.legend(loc="upper left")
    plt.show()


def line_str(m, b):
    """cones = 12.62 × temp − 637.6"""
    return f"cones = {m:.2f} × temp {'−' if b < 0 else '+'} {abs(b):.1f}"


def check(ok, good, bad):
    print(("✅ " + good) if ok else ("❌ " + bad))


print("✅ Setup done — 12 days of ice-cream data loaded.")

## 2.1 · Be the string
A line is just **two numbers**: the **slope** (extra cones per extra degree) and the
**intercept** (where it starts). Move them and watch the score, the **total of the squares** —
lower is better.

In [ ]:
# ✍️ YOU TRY 2.1.1 — drag the sliders. Can you get the total under 5,000?
interact(draw_line,
         m=FloatSlider(value=10, min=0, max=20, step=0.1, description="slope m", continuous_update=False),
         b=FloatSlider(value=-500, min=-1000, max=200, step=10, description="intercept b", continuous_update=False),
         title=fixed("Your line"));

In [ ]:
# ✅ CHECKPOINT 2.1.2 — type your best slope and intercept into the boxes, then run this cell
my_m = 10.0   # @param {type:"number"}
my_b = -500.0 # @param {type:"number"}

score = total_squares(my_m, my_b)
print(f"Your line: {line_str(my_m, my_b)}   →   total of the squares = {score:,.0f}")
check(score < 5000, "Under 5,000. You just trained a model by hand.",
      "Not under 5,000 yet. When you change the slope, fix the intercept too.")

## 2.2 · Let the computer move the string
The computer runs the same loop — **guess → measure → adjust** — but it works out exactly which
way is downhill for each number and takes a small step that way. That is **gradient descent**.
It starts from the worst possible guess: a flat line at zero.

In [ ]:
# 🔍 READ 2.2.1 — predict first: after 100 steps, will the computer beat your best score?
m, b, history = train(lr=0.1, steps=100)
print(f"After 100 steps: {line_str(m, b)}")
print(f"Total of the squares: {history[-1]:,.0f}   (it started at {history[0]:,.0f})")
show_training(lr=0.1, moments=[0, 2, 5, 10, 30])

## 2.3 · When the line lies

In [ ]:
# 🔍 READ 2.3.1 — predict first: how many cones on a freezing 32°F day? Then move the slider.
degrees = 32  # @param {type:"slider", min:0, max:120, step:1}

m, b, _ = train(lr=0.1, steps=500)
print(f"At {degrees}°F the line predicts {m * degrees + b:,.0f} cones.")
show_prediction(degrees, m, b)

**Talk at your table:** between which temperatures would you trust this line? What does it
**not know** — weekends, rain, the shop running out?

## Now a network that has to see
The Glimmerscope sees **2×2 pictures** — each pixel from −1 (white) to +1 (black) — and has to say
**solid**, **vertical**, **horizontal** or **diagonal**. Run the network setup first.

In [ ]:
# @title ⚙️ PART 2 · NETWORK SETUP — run before 2.4, and again if Colab reconnects { display-mode: "form" }
import warnings
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPClassifier
warnings.filterwarnings("ignore")

CATEGORIES = ["solid", "vertical", "horizontal", "diagonal"]


def make_images(n, noise=0.08, seed=0):
    """n random 2×2 pictures. Pixels run from -1 (white) to +1 (black), in the order
    top-left, top-right, bottom-left, bottom-right."""
    rng = np.random.default_rng(seed)
    X, y = [], []
    for _ in range(n):
        kind = rng.integers(4)
        a = rng.uniform(-1, 1)
        b = a
        while kind != 0 and abs(a - b) < 0.8:      # two clearly different shades
            b = rng.uniform(-1, 1)
        tl, tr, bl, br = {0: (a, a, a, a), 1: (a, b, a, b),
                          2: (a, a, b, b), 3: (a, b, b, a)}[kind]
        X.append(np.clip(np.array([tl, tr, bl, br]) + rng.normal(0, noise, 4), -1, 1))
        y.append(kind)
    return np.array(X), np.array(y)


def _frame(ax):
    """No ticks, but a visible border, so white pixels don't vanish into the page."""
    ax.set_xticks([]); ax.set_yticks([])
    for side in ax.spines.values():
        side.set_visible(True); side.set_color("gray")


def show_images(X, y, predicted=None, per_row=8):
    n = len(X)
    rows = (n + per_row - 1) // per_row
    fig, axes = plt.subplots(rows, per_row, figsize=(1.6 * per_row, 1.9 * rows))
    for ax, i in zip(np.ravel(axes), range(rows * per_row)):
        ax.axis("off")
        if i >= n:
            continue
        ax.imshow(X[i].reshape(2, 2), cmap="gray_r", vmin=-1, vmax=1)
        _frame(ax)
        label = CATEGORIES[y[i]]
        if predicted is not None and predicted[i] != y[i]:
            ax.set_title(f"✗ {CATEGORIES[predicted[i]]}", color="crimson", fontsize=10)
        else:
            ax.set_title(label, fontsize=10)
    plt.tight_layout(); plt.show()


def show_guess(net, pixels):
    probs = net.predict_proba([pixels])[0]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3), gridspec_kw={"width_ratios": [1, 2.5]})
    a1.imshow(np.array(pixels).reshape(2, 2), cmap="gray_r", vmin=-1, vmax=1); _frame(a1)
    a1.set_title("your picture")
    a2.barh(CATEGORIES, probs, color=["crimson" if p == probs.max() else "lightgray" for p in probs])
    a2.set_xlim(0, 1); a2.set_title("how sure the network is")
    plt.tight_layout(); plt.show()


def check(ok, good, bad):
    print(("✅ " + good) if ok else ("❌ " + bad))


X_train, y_train = make_images(400, seed=1)
X_test, y_test = make_images(200, seed=2)      # pictures the network never sees while learning
print("✅ Setup done — 400 practice pictures and 200 test pictures made.")

## 2.4 · Train it, test it, fool it

In [ ]:
# ▶ DEMO 2.4.1 — sixteen practice pictures, with their right answers
show_images(X_train[:16], y_train[:16])

In [ ]:
# ▶ DEMO 2.4.2 — 4 pixels in → 8 hidden neurons → 4 answers out, trained with the same loop.
# Then tested on new pictures it never saw: any it gets wrong are marked ✗ in red.
net = MLPClassifier(hidden_layer_sizes=(8,), activation="relu", max_iter=3000, random_state=0)
net.fit(X_train, y_train)
print(f"right on the practice pictures: {net.score(X_train, y_train):.0%}")
print(f"right on NEW pictures:          {net.score(X_test, y_test):.0%}")
show_images(X_test[:16], y_test[:16], predicted=net.predict(X_test[:16]))

In [ ]:
# ✍️ YOU TRY 2.4.3 — draw your own picture (-1 white … +1 black). Try to fool it.
top_left     = 1.0   # @param {type:"slider", min:-1, max:1, step:0.1}
top_right    = 1.0   # @param {type:"slider", min:-1, max:1, step:0.1}
bottom_left  = -1.0  # @param {type:"slider", min:-1, max:1, step:0.1}
bottom_right = -1.0  # @param {type:"slider", min:-1, max:1, step:0.1}

show_guess(net, [top_left, top_right, bottom_left, bottom_right])

## 2.5 · Too little to learn from

In [ ]:
# ✍️ YOU TRY 2.5.1 — a big network with only a few practice pictures. Then fix it.
practice_pictures = 8   # @param {type:"integer"}

big = MLPClassifier(hidden_layer_sizes=(64,), activation="relu", max_iter=3000, random_state=0)
big.fit(X_train[:practice_pictures], y_train[:practice_pictures])
on_practice, on_new = big.score(X_train[:practice_pictures], y_train[:practice_pictures]), big.score(X_test, y_test)
print(f"practice pictures: {on_practice:.0%}   new pictures: {on_new:.0%}")
check(on_new >= 0.9, "Now it generalises.",
      "It memorised the practice pictures but can't handle new ones — that's overfitting. Give it more pictures.")

## Part 2 wrap
**Training** finds the numbers that make the loss smallest. A model is only trustworthy **where it
has seen examples** — and one that is perfect on practice but poor on anything new has
**overfitted**.

**Next — D01.03:** the same networks, scaled up to read text.

# Part 3 · How an LLM Reads
**Deck D01.03** · 11:15

Three things every LLM does: chop text into **tokens**, predict the **next token**, and turn meaning into numbers (**embeddings**).

In [ ]:
# @title ⚙️ PART 3 SETUP — run this first, and again if Colab reconnects (about a minute) { display-mode: "form" }
import importlib.util, subprocess, sys, warnings
warnings.filterwarnings("ignore")
if importlib.util.find_spec("sentence_transformers") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"], check=True)

import numpy as np
import matplotlib.pyplot as plt
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer

# A small, old LLM (2019, 82 million numbers). Today's chat models are thousands of times bigger,
# but they read and predict in exactly this way.
tok = AutoTokenizer.from_pretrained("distilgpt2")
llm = AutoModelForCausalLM.from_pretrained("distilgpt2").eval()

# An embedding model: turns any sentence into 384 numbers that capture its meaning.
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


def show_tokens(text):
    ids = tok.encode(text)
    pieces = [tok.decode([i]) for i in ids]
    print(f"{len(pieces)} tokens:  " + " ".join(f"[{p}]" for p in pieces))


def next_words(text, k=10):
    """The model's top-k guesses for the next token, with probabilities."""
    with torch.no_grad():
        logits = llm(**tok(text, return_tensors="pt")).logits[0, -1]
    probs = torch.softmax(logits, dim=-1)
    top = torch.topk(probs, k)
    words = [tok.decode([int(i)]) for i in top.indices]
    return words, [float(p) for p in top.values]


def show_next(text, k=10):
    words, probs = next_words(text, k)
    plt.figure(figsize=(8, 0.4 * k + 1))
    plt.barh([repr(w) for w in words][::-1], probs[::-1], color="steelblue")
    plt.xlabel("probability"); plt.title(f"after: \"{text}\"", fontsize=11)
    plt.show()


def similarity(texts):
    v = embedder.encode(texts, normalize_embeddings=True)
    return v @ v.T


def heatmap(texts):
    s = similarity(texts)
    fig, ax = plt.subplots(figsize=(10, 8.5))
    im = ax.imshow(s, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(texts))); ax.set_xticklabels(texts, rotation=60, ha="right")
    ax.set_yticks(range(len(texts))); ax.set_yticklabels(texts)
    for i in range(len(texts)):
        for j in range(len(texts)):
            ax.text(j, i, f"{s[i, j]:.2f}", ha="center", va="center", fontsize=8,
                    color="white" if s[i, j] > 0.6 else "black")
    plt.colorbar(im, label="how alike (1 = same meaning)"); plt.tight_layout(); plt.show()


def meaning_map(texts, highlight=()):
    """Squash 384 numbers down to 2 so we can look at them (PCA)."""
    v = embedder.encode(texts, normalize_embeddings=True)
    v = v - v.mean(axis=0)
    xy = v @ np.linalg.svd(v, full_matrices=False)[2][:2].T
    plt.figure(figsize=(10, 7))
    for k, ((x, y), t) in enumerate(zip(xy, texts)):
        mine = t in highlight
        plt.scatter(x, y, s=120 if mine else 60, color="crimson" if mine else "steelblue")
        plt.annotate(t, (x, y), xytext=(6, 4 if k % 2 else -12), textcoords="offset points",
                     fontsize=11, weight="bold" if mine else "normal")
    plt.xticks([]); plt.yticks([]); plt.title("the meaning map (384 numbers squashed to 2)")
    plt.show()


def check(ok, good, bad):
    print(("✅ " + good) if ok else ("❌ " + bad))


print("✅ Setup done — an LLM and an embedding model are loaded.")

## 3.1 · Tokens

An LLM never sees letters or words. It sees **tokens** — common chunks of text, each with an ID
number. Common words are one token; rare words get split up.

In [ ]:
# ▶ DEMO 3.1.1 — how text gets chopped up
for text in ["The ice cream sold out.",
             "unbelievably",
             "decarbonisation",
             "https://www.ucsd.edu",
             "Guten Morgen, wie geht's?"]:
    show_tokens(text)

In [ ]:
# ✍️ YOU TRY 3.1.2 — your name, your company, a word from your field. How many tokens?
my_text = "Convergent Systems Engineering"  # @param {type:"string"}
show_tokens(my_text)

Rare words, names and other languages cost **more tokens**. That matters: LLMs are priced and
limited by tokens, not words.

## 3.2 · Predict the next token

That's the whole job of an LLM during training: read some text, guess the next token, see how
wrong it was, adjust. **The loop**, on most of the internet.

In [ ]:
# 🔍 READ 3.2.1 — predict first: what's the most likely next word? Write it down, then run.
show_next("The ice cream shop sold out because it was very")

In [ ]:
# ✍️ YOU TRY 3.2.2 — your own unfinished sentence
my_sentence = "To reduce our carbon emissions, the company will"  # @param {type:"string"}
show_next(my_sentence)

An LLM writes by doing this over and over: predict, pick a word, add it, predict again. It is
producing **likely text, not looking up facts** — keep that in mind this afternoon.

## 3.3 · Embeddings: meaning as numbers

An embedding model turns any sentence into **384 numbers**. Sentences that mean similar things
get similar numbers — even when they share no words.

In [ ]:
# 🔍 READ 3.3.1 — the twelve phrases from the slide. Predict first: which is most like
# "cut carbon emissions"? And where does "send a carbon copy" end up?
cards = ["cut carbon emissions", "reduce CO2 output", "lower our carbon footprint",
         "save water", "reduce water use", "fix leaking pipes",
         "quarterly earnings", "profit margin", "revenue growth",
         "recycle plastic", "compost food scraps",
         "send a carbon copy"]
heatmap(cards)

In [ ]:
# ✅ CHECKPOINT 3.3.2 — which phrase does the model think is closest to "cut carbon emissions"?
# Type it exactly as it appears in the list, then run.
my_answer = "send a carbon copy"  # @param {type:"string"}

s = similarity(cards)[0]
best = cards[int(np.argsort(s)[-2])]          # [-1] is the card itself
check(my_answer == best, f"Right: '{best}' ({s[cards.index(best)]:.2f}).",
      "Not that one. Look along the top row of the heatmap.")

In [ ]:
# ✍️ YOU TRY 3.3.3 — add three sentences of your own. Where do they land?
mine = ["install solar panels",   # @param {type:"string"}
        "shorter showers",        # @param {type:"string"}
        "beat analyst forecasts"] # @param {type:"string"}
meaning_map(cards + mine, highlight=mine)

**Talk at your table:** where did the model disagree with your table's grouping? Who decided what
"similar" means — and what text did the model learn that from?

## Part 3 wrap

You can now:
- say what a **token** is, and why rare words cost more
- say what an LLM actually does: **predict the next token**, again and again
- say what an **embedding** is: meaning as numbers, so similar ideas sit close together

**Next — D01.04, Build a RAG (after lunch):** search by meaning, plus next-token prediction, on a prepared public document.

# Part 4 · Build a RAG
**Deck D01.04** · 1:00

**RAG** — *retrieval-augmented generation* — means: find the passages in **your** documents that
match a question, then hand them to an AI along with the question, so it answers from your
documents instead of from memory.

You'll build every step and **see what each one produces**. The last step — an AI writing the
answer — uses **Colab's built-in AI** if your Google account has it. If it doesn't, the notebook
says so, and you paste the prompt into **your own AI** (ChatGPT, Claude, Gemini, Copilot…) instead.
No API key either way.

**Your documents:** pick your team's problem number (**1–4**) in **4.1** — the notebook downloads a
prepared public excerpt and shows three questions already tested against it. **0** is a built-in
fictional sample.

⚠️ **Public documents only.** No CUI, no ITAR/EAR technical data, nothing classified, nothing from a
program, no personnel records — this part sends text to an AI. If you're not sure it's public, it isn't.

In [ ]:
# @title ⚙️ PART 4 SETUP — run this first, and again if Colab reconnects (about a minute) { display-mode: "form" }
import html, importlib.util, os, subprocess, sys, textwrap, urllib.request, warnings
warnings.filterwarnings("ignore")
for package, module in (("sentence-transformers", "sentence_transformers"), ("pypdf", "pypdf")):
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import numpy as np
import matplotlib.pyplot as plt
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from IPython.display import HTML, display

IN_COLAB = "google.colab" in sys.modules
if "embedder" not in globals():      # Part 3 already loaded it, unless Colab reconnected
    embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

SAMPLE_NAME = "KESTREL lessons-learned report (fictional sample)"
SAMPLE = """=== page 1 ===

Program KESTREL — Lessons Learned Report, Phase B
Coastal Systems Integration Center (fictional) · Unclassified sample document

Executive summary

KESTREL integrates a new sensor and communications package onto an existing fleet of uncrewed
surface vessels. Phase B covered detailed design through first article test. The package met its
performance requirements, but the phase finished late and over budget, and most of the cost came
from problems we had seen before on earlier programs.

In short: the schedule slipped four months, rework cost $2.1 million, and three of the five
lessons below were already in our own lessons-learned database from earlier programs. We wrote
them down. We did not learn them.

This report is written for the next program team, not for the record. Each lesson ends with one
recommendation we believe would have prevented it.

How we collected these lessons

We held eleven structured interviews with the integrated product team, reviewed 214 change
requests, and compared our plan against what happened at each review. Interviews were
confidential; no names appear in this report.

=== page 2 ===

Lesson 1 — Requirements changed, and the tests did not

After the critical design review, 37 requirements changed. Nine of those changes were never
carried through to the test procedures, because the link between requirements and tests was kept
in a spreadsheet that was updated by hand before each review. At first article test, two of the
nine caused failures that took eleven weeks to trace back to their cause.

Recommendation: keep requirement-to-test links in the requirements tool itself, and treat any
requirement change without an updated test link as an open action item.

Lesson 2 — The interface lived in one person's head

The legacy vessel's power-and-data interface was documented in 2009 and never updated. The only
engineer who understood its undocumented behaviour retired in the second month of Phase B. Two
contractors spent seven weeks reverse-engineering what one conversation could have told them.

Recommendation: before a key engineer leaves, schedule recorded walkthroughs of anything they own
that is not fully documented, and budget the time for it.

Lesson 3 — A part went end-of-life mid-design

The processing board's main programmable chip was announced end-of-life four months into Phase B.
We made a lifetime buy of 1,200 units, enough for the planned fleet plus 15% spares, at a cost of
$640,000. A redesign for a newer chip is planned for a later phase.

Recommendation: check the obsolescence status of every critical part at each design review, not
only at the start of the program.

=== page 3 ===

Lesson 4 — The AI assistant was right in the demo

We piloted an AI assistant that summarises change requests for engineers. In the demonstration it
was accurate on every example. In daily use it summarised against an out-of-date copy of the
requirements database, and twice gave confident answers based on requirements that had already
been changed. No harm resulted, because an engineer checked each summary before acting on it.

Recommendation: continue the pilot only with a human approving every summary, and with a check
that the assistant is reading the current database before each use.

Lesson 5 — A marking mistake on a shared drawing set

A drawing set was shared with a subcontractor before its controlled-information markings had
been reviewed. The error was caught within a day and corrected, and no information left approved
systems. It still cost a week of reviews and a formal report.

Recommendation: review markings before any document leaves the team, not after.

Timeline

| Milestone | Planned | Actual |
| --- | --- | --- |
| Critical design review | Month 3 | Month 3 |
| Test procedures complete | Month 7 | Month 10 |
| First article test | Month 9 | Month 15 |

Measured from the critical design review to first article test, Phase B finished six months late.

=== page 4 ===

What we would do differently

Most of our problems were not technical. They came from defaults nobody questioned: that a
spreadsheet would stay up to date, that the documents held what our people knew, that a part
available at the start would stay available, and that a tool that worked in a demonstration would
work in daily use.

Next steps for Phase C

- Move requirement-to-test traceability into the requirements tool before Phase C begins
- Record walkthroughs of the three remaining interfaces that depend on a single engineer
- Add an obsolescence check to every design review checklist
- Re-run the AI assistant pilot for three months, with the safeguards in Lesson 4
- Add a markings review to the document-release checklist"""
PROBLEMS_URL = "https://raw.githubusercontent.com/cose-2026/220/main/problems/"
PROBLEMS = {'0': {'title': 'KESTREL lessons-learned report (fictional sample)',
       'file': None,
       'source': 'Fictional, written for CoSE 220 — built into the notebook',
       'default': 'Lessons learned get written down, so they get learned.',
       'questions': [{'kind': 'answered',
                      'q': 'How many requirements changed after the critical design review?',
                      'expect': '37 (page 2).'},
                     {'kind': 'not answered',
                      'q': "What is the program's total contract value?",
                      'expect': "The report doesn't say. A good answer says so."},
                     {'kind': 'tricky',
                      'q': 'How many months did the schedule slip?',
                      'expect': 'The report contradicts itself: the summary says four months (page 1), the '
                                'timeline shows six (page 3). A good answer flags both.'}],
       'decision': 'A program manager starting Phase C needs to decide which lesson to act on first, using '
                   'this report.'},
 '1': {'title': 'NASA Systems Engineering Handbook — 6.2 Requirements Management',
       'file': '1-traceability-nasa-se-handbook.pdf',
       'source': 'NASA SP-2016-6105 Rev 2 (public), pages 1 and 139–149',
       'default': 'Traceability is a spreadsheet someone updates before the review.',
       'questions': [{'kind': 'answered',
                      'q': "When should a requirement's traceability be recorded?",
                      'expect': 'As each requirement is documented — not later (page 5).'},
                     {'kind': 'not answered',
                      'q': 'Which requirements management tool does NASA recommend?',
                      'expect': 'None is named. Page 5 mentions only a requirements matrix or a modeling '
                                'application.'},
                     {'kind': 'tricky',
                      'q': 'Must every requirement trace to a parent requirement?',
                      'expect': 'No: it can be self-derived if the next level up agrees. Otherwise the trace '
                                "is flawed or the requirement is 'gold plating' (page 5)."}],
       'decision': 'A NIWC systems engineer needs to decide whether an unparented requirement should be '
                   'kept, re-traced or cut before the review, using this handbook.'},
 '2': {'title': "NASA OIG audit of NASA's lessons-learned process (IG-12-012)",
       'file': '2-lessons-learned-nasa-oig.pdf',
       'source': 'NASA Office of Inspector General, IG-12-012 (public), pages 1–30',
       'default': 'Lessons learned get written down, so they get learned.',
       'questions': [{'kind': 'answered',
                      'q': 'How many surveyed project managers contributed lessons to LLIS?',
                      'expect': '12 of 28, or 43 percent (page 5).'},
                     {'kind': 'not answered',
                      'q': 'Did LLIS use improve after NASA appointed a Chief Knowledge Officer?',
                      'expect': "The report doesn't say: it predates the appointment — a Chief Knowledge "
                                "Officer 'will be assigned', due March 2013 (pages 7, 26)."},
                     {'kind': 'tricky',
                      'q': 'Does NASA require lessons learned before project closeout?',
                      'expect': "Not under the binding policy: NPR 7120.5D doesn't require them until "
                                'closeout (page 5). A 2009 letter urged capturing them right after each '
                                'effort (page 16), but it was informal and brought no measurable improvement '
                                '(page 6).'}],
       'decision': 'A program manager needs to decide when the team captures lessons — at each milestone or '
                   'at closeout — using this audit.'},
 '3': {'title': 'NIST AI Risk Management Framework 1.0',
       'file': '3-ai-trust-nist-ai-rmf.pdf',
       'source': 'NIST AI 100-1, January 2023 (public), pages 1–24',
       'default': "If the AI is right in the demo, it's ready for operators.",
       'questions': [{'kind': 'answered',
                      'q': 'Do lab test results reflect operational risk?',
                      'expect': 'Not necessarily — they may differ from risks that emerge in real-world '
                                'settings (page 11).'},
                     {'kind': 'not answered',
                      'q': 'What accuracy threshold must an AI meet before deployment?',
                      'expect': "None is given: the RMF doesn't prescribe risk tolerance (page 12) and "
                                'leaves threshold values to human judgment (page 17).'},
                     {'kind': 'tricky',
                      'q': 'If the AI is accurate in testing, is it trustworthy?',
                      'expect': 'No. Valid and reliable is necessary but not sufficient — trust is only as '
                                'strong as the weakest characteristic (page 17) — and accuracy must be '
                                'paired with realistic test sets (page 19).'}],
       'decision': 'A test-and-evaluation lead needs to make the go/no-go call on fielding an AI aid after a '
                   'successful demo, using the NIST AI RMF.'},
 '4': {'title': '40 CFR 60 Subpart KKKK — stationary combustion turbines (2024 edition)',
       'file': '4-turbine-emissions-40cfr60-kkkk.pdf',
       'source': 'Code of Federal Regulations, 2024 edition via govinfo.gov (public), all 16 pages',
       'default': "The emissions rule we were permitted under is the one we'll always meet.",
       'questions': [{'kind': 'answered',
                      'q': 'What NOx limit applies to a new natural-gas turbine above 850 MMBtu/h?',
                      'expect': '15 ppm at 15 percent O2, or 54 ng/J of useful output (0.43 lb/MWh) — Table '
                                '1, page 15.'},
                     {'kind': 'not answered',
                      'q': 'What NOx limits does Subpart KKKKa set?',
                      'expect': 'Not in this document: it is the 2024 edition, and Subpart KKKKa came '
                                'later.'},
                     {'kind': 'tricky',
                      'q': 'Is our turbine test stand exempt from KKKK NOx limits?',
                      'expect': 'Test cells/stands are exempt from the whole subpart (page 2); R&D turbines '
                                'can be exempted from the NOx limit only by petition, case by case (page '
                                '10).'}],
       'decision': "A gas-turbine combustion engineer needs to decide whether a new large natural-gas unit's "
                   'NOx design margin meets the federal limit, using this regulation.'}}


def read_pdf(path, name):
    pages = []
    for n, page in enumerate(PdfReader(path).pages, start=1):
        text = " ".join((page.extract_text() or "").split())
        if text:
            pages.append({"source": name, "page": n, "text": text})
    return pages


def sample_pages():
    pages = []
    for part in SAMPLE.split("=== page ")[1:]:
        n, text = part.split(" ===", 1)
        pages.append({"source": SAMPLE_NAME, "page": int(n), "text": " ".join(text.split())})
    return pages


def get_documents(problem):
    """0 = the built-in sample · 1-4 = a prepared public excerpt. None if the download failed."""
    info = PROBLEMS.get(str(problem))
    if problem == 0 or not info:
        return sample_pages()
    os.makedirs("problems", exist_ok=True)
    path = os.path.join("problems", info["file"])
    if not os.path.exists(path):
        try:
            urllib.request.urlretrieve(PROBLEMS_URL + info["file"], path)
        except Exception as err:
            print(f"❌ Couldn't download problem {problem}'s document ({err}).")
            if os.path.exists(path):
                os.remove(path)
            return None
    return read_pdf(path, info["title"])


def show_problem(problem):
    info = PROBLEMS.get(str(problem), PROBLEMS["0"])
    print(f"PROBLEM {problem}: {info['title']}")
    print(f"Source: {info['source']}")
    print(f"Default to question: \"{info['default']}\"")
    print(f"\nExample decision: {info['decision']}\n")
    print("Three tested questions:")
    for n, q in enumerate(info["questions"], start=1):
        print(f"  {n}. ({q['kind']}) {q['q']}")


def make_chunks(pages, words_per_chunk, overlap):
    """Cut each page into overlapping runs of words. Each chunk remembers where it came from."""
    chunks = []
    step = max(1, words_per_chunk - overlap)
    for p in pages:
        words = p["text"].split()
        for start in range(0, max(1, len(words) - overlap), step):
            piece = " ".join(words[start:start + words_per_chunk])
            if piece:
                chunks.append({"source": p["source"], "page": p["page"], "text": piece})
    return chunks


def search(question, chunks, vectors, k):
    q = embedder.encode([question], normalize_embeddings=True)[0]
    scores = vectors @ q
    best = np.argsort(scores)[::-1][:k]
    return [dict(chunks[i], score=float(scores[i])) for i in best]


def show_hits(hits):
    for n, h in enumerate(hits, start=1):
        print(f"[{n}]  score {h['score']:.2f}   {h['source']}, page {h['page']}")
        print(textwrap.fill(h["text"], 100, initial_indent="     ", subsequent_indent="     "))
        print()


def build_prompt(question, hits, strict=True, audience="", extra_rule=""):
    passages = "\n\n".join(f"[{n}] ({h['source']}, page {h['page']})\n{h['text']}"
                             for n, h in enumerate(hits, start=1))
    rules = ("Answer using ONLY the numbered passages below. If they do not contain the answer, "
             "reply exactly: The documents don't say.\nAfter each fact, cite the passage number "
             "you used, like [2].") if strict else "Use the passages below to help you answer."
    who = f"You are an assistant for {audience}, answering a question about some documents." if audience \
        else "You are answering a question about some documents."
    extra = f"\n{extra_rule}" if extra_rule else ""
    return f"{who}\n{rules}{extra}\n\nPASSAGES\n\n{passages}\n\nQUESTION: {question}"


def show_prompt(prompt):
    """A box to copy from. Click in it, select all, copy, paste into your AI."""
    if IN_COLAB:
        display(HTML(
            "<p><b>Click in the box → select all (Ctrl-A / ⌘-A) → copy (Ctrl-C / ⌘-C) → paste into your AI.</b></p>"
            f"<textarea readonly rows='16' style='width:100%;font-family:monospace;font-size:12px' "
            f"onclick='this.select()'>{html.escape(prompt)}</textarea>"))
    else:
        print(prompt)
    print(f"\n({len(prompt.split())} words, roughly {len(prompt) // 4} tokens)")


# Colab's built-in AI: no API key. A preview feature, so it may be missing or rate-limited on some
# accounts -- then ask_ai() returns None and you copy and paste the prompt instead.
try:
    from google.colab import ai as colab_ai
except Exception:
    colab_ai = None


def ask_ai(prompt):
    """Colab's built-in AI's answer, or None if it isn't available here."""
    if colab_ai is None:
        return None
    try:
        return colab_ai.generate_text(prompt).strip()
    except Exception as err:
        print(f"(Colab's built-in AI didn't answer: {err})")
        return None


def answer_or_copy(prompt):
    """Ask Colab's AI; if it can't, show the prompt to copy into your own AI."""
    reply = ask_ai(prompt)
    if reply:
        print("COLAB'S AI ANSWERED:\n")
        print(textwrap.fill(reply, 100))
    else:
        print("Colab's built-in AI isn't available — copy this prompt into your own AI instead.\n")
        show_prompt(prompt)


def check(ok, good, bad):
    print(("✅ " + good) if ok else ("❌ " + bad))


print("✅ Setup done — the embedding model is loaded.")
print("✅ Colab's built-in AI is available." if colab_ai else
      "ℹ️  Colab's built-in AI isn't available here — you'll copy and paste prompts into your own AI.")

## 4.1 · Load

In [ ]:
# ✍️ YOU TRY 4.1.1 — pick your team's problem (0 = the sample · 1-4 = prepared), then run
problem = 0  # @param [0, 1, 2, 3, 4] {type:"raw"}

pages = get_documents(problem)
if pages is None:
    problem = 0
    pages = get_documents(0)
    print("❌ SWITCHED TO PROBLEM 0, the built-in sample, with its own questions. Tell your instructor.\n")
loaded_problem = problem
show_problem(problem)
print(f"\n{len(pages)} pages, {sum(len(p['text'].split()) for p in pages):,} words. The start:\n")
print(textwrap.fill(pages[0]["text"][:500] + " …", 100))

## 4.2 · Chunk and embed
An AI can only read so much at once, so we cut the document into **chunks** — short runs of
words — and turn each one into 384 numbers, the same embeddings as Part 3.

In [ ]:
# ▶ DEMO 4.2.1 — cut into chunks, then turn every chunk into numbers (re-run if you change the size)
words_per_chunk = 150  # @param {type:"slider", min:40, max:400, step:10}
overlap = 30

chunks = make_chunks(pages, words_per_chunk, overlap)
vectors = embedder.encode([c["text"] for c in chunks], normalize_embeddings=True)
print(f"{len(chunks)} chunks of up to {words_per_chunk} words, each now {vectors.shape[1]} numbers.\n")
print(f"— the first chunk ({chunks[0]['source']}, page {chunks[0]['page']}) —")
print(textwrap.fill(chunks[0]["text"], 100))

## 4.3 · Retrieve
Turn the question into numbers the same way, and find the chunks whose numbers are closest.

In [ ]:
# ✍️ YOU TRY 4.3.1 — ask a question. Leave the box empty to use your problem's first tested question.
question = ""  # @param {type:"string"}
top_k = 3  # @param {type:"slider", min:1, max:8, step:1}

if not question.strip():
    question = PROBLEMS.get(str(problem), PROBLEMS["0"])["questions"][0]["q"]
print("QUESTION:", question, "\n")
hits = search(question, chunks, vectors, top_k)
show_hits(hits)

**Look before you go on:** is the answer actually in one of those passages? If not, nothing later
can fix it.

## 4.4 · Ask the AI
This is the whole trick of RAG: it's **just a prompt** — some rules, the passages, and the question.
If Colab's built-in AI isn't available on your account, you get a box to copy into your own AI.

In [ ]:
# ▶ DEMO 4.4.1 — build the prompt and get the answer
prompt = build_prompt(question, hits)
print(prompt[:600], "…\n")
answer_or_copy(prompt)

**You built a RAG:** load → chunk → embed → retrieve → prompt → answer. Every "chat with your
documents" product is this pipeline, with more polish. Did the answer cite a passage like [1] —
and does that passage really say it?

**Next — Part 5:** your three tested questions.

# Part 5 · Diagnose and Retest
**Deck D01.05** · 2:15

One good answer proves nothing. Ask **three tested questions**, check each wrong answer with **three
yes/no checks**, change **one** thing, and ask again.

In [ ]:
# @title ⚙️ PART 5 SETUP — run this first, and again if Colab reconnects { display-mode: "form" }
# Part 5 uses the RAG from Part 4. After a reconnect, pick your problem again in the box, then run.
problem_again = "same as Part 4"  # @param ["same as Part 4", "0", "1", "2", "3", "4"]

if "get_documents" not in globals():
    print("❌ Scroll up and run the PART 4 SETUP cell first, then run this one again.")
elif problem_again == "same as Part 4" and "loaded_problem" not in globals():
    print("❌ Colab reconnected, so it forgot your problem. Pick your problem number in the box, then run again.")
else:
    problem = loaded_problem if problem_again == "same as Part 4" else int(problem_again)
    if globals().get("loaded_problem") != problem or "vectors" not in globals():
        words_per_chunk, overlap, top_k = 150, 30, 3
        pages = get_documents(problem)
        if pages is None:
            problem, pages = 0, get_documents(0)
            print("❌ SWITCHED TO PROBLEM 0, the built-in sample, with its own questions. Tell your instructor.")
        chunks = make_chunks(pages, words_per_chunk, overlap)
        vectors = embedder.encode([c["text"] for c in chunks], normalize_embeddings=True)
        loaded_problem = problem
        print(f"Rebuilt the RAG for problem {problem} (150-word chunks, top_k 3).")
    tested = PROBLEMS[str(problem)]["questions"]
    print(f"✅ Ready — problem {problem}: {PROBLEMS[str(problem)]['title']}")

## 5.1 · Three questions
Your problem comes with three questions, and we know what the document really says:
**answered** (it's in there) · **not answered** (the right reply is *"The documents don't say"*) ·
**tricky** (needs two passages, has an exception, or the document disagrees with itself).

In [ ]:
# ✍️ YOU TRY 5.1.1 — pick question 1, 2 or 3 and run. Do all three.
question_number = 1  # @param [1, 2, 3] {type:"raw"}

q = tested[question_number - 1]
print(f"Q{question_number} ({q['kind']}): {q['q']}")
print(textwrap.fill(f"What the document really says: {q['expect']}", 100), "\n")
hits = search(q["q"], chunks, vectors, top_k)
show_hits(hits)
answer_or_copy(build_prompt(q["q"], hits))

## 5.2 · Why was it wrong?
For each wrong answer, three yes/no checks, in order. **Point to the passage** for each one.

| Check | If **no** | Try |
| --- | --- | --- |
| **1. Is the answer in the document at all?** (see *what the document really says*) | a good answer says *"The documents don't say"*. If it guessed instead… | a rule: *"If the passages don't answer it, say so. Do not guess."* |
| **2. Were the passages with the answer retrieved?** | retrieval missed it | a bigger `top_k`, or reword the question |
| **3. Did the answer use all the relevant passages, and nothing else?** | it picked one side, or added its own | a rule, e.g. *"If the passages disagree, say so and quote both."* |

In [ ]:
# ✍️ YOU TRY 5.2.1 — change ONE thing and ask again
retest_question = 3  # @param [1, 2, 3] {type:"raw"}
new_top_k = 3  # @param {type:"slider", min:1, max:8, step:1}
new_rule = ""  # @param {type:"string"}

q = tested[retest_question - 1]["q"]
print("RETEST:", q, "\n")
hits = search(q, chunks, vectors, new_top_k)
show_hits(hits)
answer_or_copy(build_prompt(q, hits, extra_rule=new_rule))

## 5.3 · Your demo card
This is what you show at 3:30. **Before you pick a verdict:** open your PDF and search it (Ctrl-F /
⌘-F) for one of your questions. Faster or slower than your RAG? Better or worse? A well-supported
**"AI isn't the right tool here"** counts just as much as a working assistant.

In [ ]:
# ✅ CHECKPOINT 5.3.1 — fill in the five boxes, then run
who_and_decision = ""  # @param {type:"string"}
it_got_right = ""  # @param {type:"string"}
it_got_wrong_and_why = ""  # @param {type:"string"}
we_changed = ""  # @param {type:"string"}
our_verdict = ""  # @param ["", "AI helps", "AI helps, with conditions", "searching the PDF or a better process would do better"]

card = [("Who and what", who_and_decision), ("Got right", it_got_right),
        ("Got wrong", it_got_wrong_and_why), ("We changed", we_changed), ("Verdict", our_verdict)]
for label, value in card:
    print(f"{label:>12}:  {value or '(fill in)'}")
check(all(v for _, v in card), "Ready for the demo.", "Fill in every box before the demos.")